In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..") # Add s higher directory to python modules path.
from environment.environment import *
from engine.agent.base_agent import *
from engine.policy.base_policy import *
np.set_printoptions(suppress=True)

In [2]:
from tensorflow.keras.datasets import cifar10
from tensorflow.keras import *
import os

In [5]:
# The data, split between train and test sets:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
print('x_train shape:', x_train.shape)
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

x_train shape: (50000, 32, 32, 3)
50000 train samples
10000 test samples


In [6]:
y_test = tf.keras.utils.to_categorical(y_test)
y_train = tf.keras.utils.to_categorical(y_train)
y_test.shape, y_train.shape

((10000, 10), (50000, 10))

In [19]:
class ToyConvNet(Model):
    def __init__(self):
        super().__init__()
        filters = [16, 32, 64, 64, 128]
        kernels = [5,  5,  3,  3,  1]
        strides = [1,  1,  1,  1,  1]
        padding = ['same', 'same', 'same', 'same', 'same']
        self.ly = []
        for f, k, s, p in zip(filters, kernels, strides, padding):
            self.ly.append(Conv2D(filters=f, kernel_size=k, strides=s, padding=p))
            self.ly.append(BatchNormalization())
            self.ly.append(Activation('relu'))
        self.ly.append(Flatten())
        self.ly.append(Dense(256, activation='relu'))
        self.ly.append(BatchNormalization())
        self.ly.append(Dense(10, activation='softmax'))

    def call(self, x):
        for f in self.ly:
            x = f(x)
        return x
    
    def get_model(self):
        x = Input(shape=(32,32,3))
        return Model(inputs=[x], outputs=self.call(x), name='ToyConvNet')

In [20]:
model = ToyConvNet().get_model()
model.summary()

Model: "ToyConvNet"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_2 (InputLayer)         [(None, 32, 32, 3)]       0         
_________________________________________________________________
conv2d_5 (Conv2D)            (None, 32, 32, 16)        1216      
_________________________________________________________________
batch_normalization_6 (Batch (None, 32, 32, 16)        64        
_________________________________________________________________
activation_5 (Activation)    (None, 32, 32, 16)        0         
_________________________________________________________________
conv2d_6 (Conv2D)            (None, 32, 32, 32)        12832     
_________________________________________________________________
batch_normalization_7 (Batch (None, 32, 32, 32)        128       
_________________________________________________________________
activation_6 (Activation)    (None, 32, 32, 32)        0

In [22]:
adam = optimizers.Adam(lr=1e-3)
model.compile(loss='categorical_crossentropy', optimizers=adam)
model.fit(x_train, y_train, batch_size=1024, epochs=20, validation_data=(x_test,y_test), shuffle=True)

Train on 50000 samples, validate on 10000 samples
Epoch 1/20
 8192/50000 [===>..........................] - ETA: 4:59 - loss: 1.8485

KeyboardInterrupt: 

In [7]:
def get_resnet_k(k = 19):
    '''
    Return model with K ResNet Blocks
    '''

    inp = Input(shape=(5,5,3))
    
    #first conv block
    x = Conv2D(filters=256, kernel_size=(3,3), strides=1, padding='same')(inp)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    #resnet blocks (19?)
    for i in range(k//2):
        x = Conv2D(filters=256, kernel_size=(3,3), strides=1, padding='same')(x)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)
        x2 = Conv2D(filters=256, kernel_size=(3,3), strides=1, padding='same')(x)
        x = x2 + x #residual
        x = BatchNormalization()(x)
        x = Activation('relu')(x)
        if i == 4:
            x = MaxPooling2D(pool_size=2)(x)

    p = Conv2D(filters=2, kernel_size=(1,1), strides=1, padding='same')(x)
    p = Flatten()(p)
    p = BatchNormalization()(p)
    p = Activation('relu')(p)
    p = Dense(128, activation='softmax')(p)

    v = Conv2D(filters=2, kernel_size=(1,1), strides=1, padding='same')(x)
    v = Flatten()(v)
    v = BatchNormalization()(v)
    v = Activation('relu')(v)
    v = Dense(256, activation='relu')(v)
    v = Dense(1, activation='tanh')(v)

    return Model(inputs=[inp], outputs=[p, v], name='ResNet{}'.format(k))

In [8]:
model = get_resnet_k()
model.summary()

Model: "ResNet19"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 5, 5, 3)]    0                                            
__________________________________________________________________________________________________
conv2d (Conv2D)                 (None, 5, 5, 256)    7168        input_1[0][0]                    
__________________________________________________________________________________________________
batch_normalization (BatchNorma (None, 5, 5, 256)    1024        conv2d[0][0]                     
__________________________________________________________________________________________________
activation (Activation)         (None, 5, 5, 256)    0           batch_normalization[0][0]        
___________________________________________________________________________________________

In [9]:
env = Santorini()
env.get_converted_board()

array([[[ 0.,  0.,  0.],
        [ 0.,  0.,  0.],
        [ 0., -1.,  0.],
        [ 0.,  0.,  0.],
        [ 0.,  0.,  0.]],

       [[ 0.,  0.,  0.],
        [ 0.,  0., 22.],
        [ 0.,  0.,  0.],
        [ 0.,  0.,  0.],
        [ 0.,  0.,  0.]],

       [[ 0.,  1.,  0.],
        [ 0.,  0.,  0.],
        [ 0.,  0., 18.],
        [ 0.,  0.,  0.],
        [ 0.,  2.,  0.]],

       [[ 0.,  0.,  0.],
        [ 0.,  0.,  0.],
        [ 0.,  0.,  0.],
        [ 0.,  0., 14.],
        [ 0.,  0.,  0.]],

       [[ 0.,  0.,  0.],
        [ 0.,  0.,  0.],
        [ 0., -2.,  0.],
        [ 0.,  0.,  0.],
        [ 0.,  0., 18.]]])

In [11]:
model.predict(env.get_converted_board().reshape(1, 5, 5, 3))

[array([[0.00779731, 0.00774884, 0.00784681, 0.00782673, 0.00786118,
         0.00778293, 0.00784156, 0.00787748, 0.00789533, 0.00792447,
         0.00781039, 0.00786444, 0.00783721, 0.00791672, 0.00773716,
         0.00773788, 0.00788496, 0.00780266, 0.00782141, 0.00779444,
         0.00790799, 0.00772601, 0.0077009 , 0.00775626, 0.00794027,
         0.0077578 , 0.00779321, 0.0078706 , 0.00786931, 0.00794603,
         0.00782612, 0.00775783, 0.00774767, 0.00777832, 0.00770803,
         0.00785671, 0.00788834, 0.00784571, 0.0076656 , 0.00790589,
         0.00775284, 0.0078271 , 0.00782254, 0.00783397, 0.00780633,
         0.00777497, 0.00773997, 0.00783067, 0.00783642, 0.00784773,
         0.00765576, 0.00784879, 0.00789336, 0.00769435, 0.00781052,
         0.00779297, 0.00782524, 0.00770439, 0.00776633, 0.00778912,
         0.00778084, 0.00771148, 0.0077859 , 0.00779313, 0.00767606,
         0.00776889, 0.00768415, 0.00788455, 0.00784798, 0.00778395,
         0.00779842, 0.00776765, 0

In [12]:
env = Santorini()
_a = np.ravel(env.get_converted_board()).astype('int32')
_a

array([ 0,  0,  0,  0,  0,  0,  0, -1,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0, 22,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,
        0,  0,  0,  0, 18,  0,  0,  0,  0,  2,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0, 14,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, -2,
        0,  0,  0,  0,  0,  0, 18], dtype=int32)

In [13]:
st = ''
for num in _a:
    st = st + str(num)
st

'0000000-100000000000022000000000010000001800002000000000000140000000000-200000018'

In [14]:
env.legal_moves()

[27,
 28,
 29,
 30,
 31,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 96,
 97,
 98,
 99,
 100]

In [16]:
env.atoi

{(-1, 'q', 'q'): 0,
 (-1, 'q', 'w'): 1,
 (-1, 'q', 'e'): 2,
 (-1, 'q', 'a'): 3,
 (-1, 'q', 'd'): 4,
 (-1, 'q', 'z'): 5,
 (-1, 'q', 'x'): 6,
 (-1, 'q', 'c'): 7,
 (-1, 'w', 'q'): 8,
 (-1, 'w', 'w'): 9,
 (-1, 'w', 'e'): 10,
 (-1, 'w', 'a'): 11,
 (-1, 'w', 'd'): 12,
 (-1, 'w', 'z'): 13,
 (-1, 'w', 'x'): 14,
 (-1, 'w', 'c'): 15,
 (-1, 'e', 'q'): 16,
 (-1, 'e', 'w'): 17,
 (-1, 'e', 'e'): 18,
 (-1, 'e', 'a'): 19,
 (-1, 'e', 'd'): 20,
 (-1, 'e', 'z'): 21,
 (-1, 'e', 'x'): 22,
 (-1, 'e', 'c'): 23,
 (-1, 'a', 'q'): 24,
 (-1, 'a', 'w'): 25,
 (-1, 'a', 'e'): 26,
 (-1, 'a', 'a'): 27,
 (-1, 'a', 'd'): 28,
 (-1, 'a', 'z'): 29,
 (-1, 'a', 'x'): 30,
 (-1, 'a', 'c'): 31,
 (-1, 'd', 'q'): 32,
 (-1, 'd', 'w'): 33,
 (-1, 'd', 'e'): 34,
 (-1, 'd', 'a'): 35,
 (-1, 'd', 'd'): 36,
 (-1, 'd', 'z'): 37,
 (-1, 'd', 'x'): 38,
 (-1, 'd', 'c'): 39,
 (-1, 'z', 'q'): 40,
 (-1, 'z', 'w'): 41,
 (-1, 'z', 'e'): 42,
 (-1, 'z', 'a'): 43,
 (-1, 'z', 'd'): 44,
 (-1, 'z', 'z'): 45,
 (-1, 'z', 'x'): 46,
 (-1, 'z', 'c'): 47,
 (